# Auditory-Evoked Pupillary Responses (AEPR) - Kaggle Deep Learning Runner

Automated GPU execution environment for training and benchmarking Deep Learning architectures on single-trial pupillometry time series.

In [ ]:
# ====================================================================
# Cell 1: Environment, Pinned Dependency Checks & GPU Verification
# ====================================================================
import sys
import os
import json
import torch
import numpy as np
import pandas as pd
import scipy
import sklearn

print('=' * 60)
print('ENVIRONMENT & HARDWARE DIAGNOSTICS')
print('=' * 60)
print(f'Python Version:       {sys.version.split()[0]}')
print(f'PyTorch Version:      {torch.__version__}')
print(f'NumPy Version:        {np.__version__}')
print(f'Pandas Version:       {pd.__version__}')
print(f'SciPy Version:        {scipy.__version__}')
print(f'Scikit-Learn Version: {sklearn.__version__}')

cuda_available = torch.cuda.is_available()
print(f'\nCUDA Available:       {cuda_available}')
if cuda_available:
    gpu_count = torch.cuda.device_count()
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU Device Name:      {gpu_name}')
    print(f'GPU Device Count:     {gpu_count}')
    print(f'GPU Total Memory:     {gpu_mem:.2f} GB')
    print(f'CUDA Version:         {torch.version.cuda}')
else:
    print('WARNING: Running in CPU mode. GPU was not detected!')
print('=' * 60)


In [ ]:
# ====================================================================
# Cell 2: Verify Mounted Kaggle Dataset
# ====================================================================
from pathlib import Path

input_dir = Path('/kaggle/input')
print(f'Mounted datasets in {input_dir}:')
for p in input_dir.glob('*'):
    print(f'  - {p.name} ({len(list(p.glob("**/*")))} files)')

aepr_dataset_dir = Path('/kaggle/input/aepr-pupillometry-dataset')
if not (aepr_dataset_dir.exists() and (aepr_dataset_dir / 'dataset_b').exists()):
    candidates = [p for p in input_dir.glob('*') if (p / 'dataset_b').exists()]
    if candidates:
        aepr_dataset_dir = candidates[0]
        print(f'Using detected dataset path: {aepr_dataset_dir}')
    else:
        print(f'Notice: standard path not found, falling back to: {input_dir}')
        aepr_dataset_dir = input_dir


In [ ]:
# ====================================================================
# Cell 3: Unpack Local Codebase (src/ and scripts/)
# ====================================================================
import base64
import tarfile
import io

payload = '''...'''

tar_bytes = base64.b64decode(payload.encode('ascii'))
with tarfile.open(fileobj=io.BytesIO(tar_bytes), mode='r:gz') as tar:
    tar.extractall(path='/kaggle/working')

if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')

print('Successfully loaded local modules into /kaggle/working.')


In [ ]:
# ====================================================================
# Cell 4: Execute Full Deep Learning Benchmarks on GPU
# ====================================================================
from scripts.run_deep_learning_benchmarks import run_all_deep_learning_benchmarks

data_path = aepr_dataset_dir
working_dir = Path('/kaggle/working')

run_all_deep_learning_benchmarks(data_dir=data_path, output_dir=working_dir)


In [ ]:
# ====================================================================
# Cell 5: Inspect Generated Reports & Figures
# ====================================================================
print('\nGenerated Outputs in /kaggle/working:')
for f in sorted(list(working_dir.rglob('*'))):
    if f.is_file():
        print(f'  - {f.relative_to(working_dir)} ({f.stat().st_size:,} bytes)')

report_file = working_dir / 'DEEP_LEARNING_REPORT.md'
if report_file.exists():
    print('\n' + '=' * 60)
    print('DEEP_LEARNING_REPORT.md HEAD:')
    print('=' * 60)
    with open(report_file) as f:
        print('\n'.join(f.readlines()[:50]))
